# VaR Backtesting, GARCH & Extreme Value Theory

**UCLA MFE 409 — Financial Risk Management (Prof. Valentin Haddad), Problem Set 4**
*Individual submission (coursework was done in an assigned study group, but each student writes up their own analysis).*

### The problem
Given a strategy's daily gains from 1/2/2014 to 12/19/2017, choose and validate a Value-at-Risk (VaR)
technique. The assignment asks for three families of approach, each backtested against realized
exceptions (days the actual loss exceeded the VaR estimate):

1. **Historical simulation** — empirical quantile of trailing returns, compared to an
   **exponentially-weighted** version that upweights recent observations (λ = 0.995).
2. **Model-building approaches** — EWMA volatility (λ = 0.96) and a GARCH(1,1) fit by
   maximum likelihood, each mapped to VaR via the normal quantile.
3. **A tail-aware approach** — normalizing returns by rolling volatility, then applying
   **Extreme Value Theory** (peaks-over-threshold, generalized Pareto tail) instead of
   assuming normal tails.

A second part applies the same tooling to a live question: estimating 2-day 99% VaR for a
$1M S&P 500 position ahead of an FOMC announcement.

> **Note on data:** the original assignment used a proprietary course-provided return series
> (`hw4_returns.csv`) that isn't included here. The code below is exactly as submitted; Part 2
> below pulls live public data and runs standalone.

## 1. Historical vs. exponentially-weighted VaR

For each day in 2015–2017, compute the historical 99% VaR (empirical 1st percentile of all
trailing returns) and an exponentially-weighted version that decays older observations by
λ = 0.995 per day.

In [ ]:
import pandas as pd
import numpy as np

data = pd.read_csv('hw4_returns.csv')
data['Date'] = pd.to_datetime(data['Date'])
data.set_index('Date', inplace=True)

lam = 0.995
new_data = data[data.index >= pd.to_datetime('2015-01-01')]
new_date = new_data.index

var_hist = []
var_exp = []

for i in new_date:
    window = data.loc[:i]
    re = window[:-1]['Return']
    var_hist.append(np.percentile(re, 1))

    n = len(re)
    time_decay = np.arange(n)[::-1]
    w = (1 - lam) / (1 - lam**n) * (lam**time_decay)

    df_temp = pd.DataFrame({'weights': w, 'return': re})
    df_sort = df_temp.sort_values(by='return', ascending=True)
    df_sort['cum_weight'] = df_sort['weights'].cumsum()

    var_exp.append(df_sort[df_sort['cum_weight'] >= 0.01]['return'].iloc[0])

results_df = pd.DataFrame({
    'Date': new_date,
    'Historical_VaR': var_hist,
    'Weighted_VaR': var_exp
})
results_df = results_df.set_index('Date')
print(results_df)

**Backtest.** Flag a day as an *exception* whenever the realized return falls below the
prior day's VaR estimate, then compare the exception rate to the 1% target implied by a 99%
VaR.

In [ ]:
results_df['Actual Returns'] = new_data['Return']
results_df['exception historical'] = np.where(
    results_df['Actual Returns'] < results_df['Historical_VaR'], 1, 0)
results_df['exception exponential'] = np.where(
    results_df['Actual Returns'] < results_df['Weighted_VaR'], 1, 0)

excep_his = sum(results_df['exception historical'])
excep_wei = sum(results_df['exception exponential'])

his_rate = round(excep_his / len(results_df), 3)
wei_rate = round(excep_wei / len(results_df), 3)

print(f'The historical method has {excep_his} exceptions, exception rate is {his_rate}.\n'
      f'The exponential method has {excep_wei} exceptions, exception rate is {wei_rate}.\n'
      f'Both exception rates are greater than 1%, so both models underestimate the risk. While '
      f'the exponential method is closer to 1%, it performs well in the market. The historical '
      f'method underestimates the risk.')

**Result:** the historical method produced 22 exceptions (2.9% exception rate) and the
exponentially-weighted method produced 13 (1.7%). Both exceed the 1% target — both models
underestimate tail risk over this sample — but the exponentially-weighted version, by
upweighting recent volatility, comes noticeably closer to correct coverage.

### 95% confidence intervals on the VaR estimate
Parametric (normal-theory) CI for the historical VaR, and bootstrap CIs for both the
historical and exponentially-weighted estimates.

In [ ]:
z, z_c = 2.326, 1.96
results = []

for i in new_date:
    rang = data.loc[:i]
    re = rang[:-1]['Return']
    mu, sig = np.mean(re), np.std(re)

    var = mu - z * sig
    se = sig * np.sqrt(1 / len(re) * (1 + z**2 / 2))
    low, high = var - z_c * se, var + z_c * se

    hist_list, exp_list = [], []
    n = len(re)
    for k in range(100):
        sample_index = np.random.randint(0, len(re), len(re))
        sample_data = re.iloc[sample_index]

        hist_list.append(np.percentile(sample_data, 1))

        time_decay = np.arange(len(sample_data))[::-1]
        w = (1 - lam) / (1 - lam**n) * (lam**time_decay)
        df_temp = pd.DataFrame({'weights': w, 'return': sample_data})
        df_boot = df_temp.sort_values(by='return', ascending=True)
        df_boot['cum_weight'] = df_boot['weights'].cumsum()
        exp_list.append(df_boot[df_boot['cum_weight'] >= 0.01]['return'].iloc[0])

    hist_low, hist_high = np.percentile(hist_list, 2.5), np.percentile(hist_list, 97.5)
    exp_low, exp_high = np.percentile(exp_list, 2.5), np.percentile(exp_list, 97.5)

    results.append({
        'Date': i, 'Param_CI_Lower': low, 'Param_CI_Upper': high,
        'Boot_Hist_CI_Lower': hist_low, 'Boot_Hist_CI_Upper': hist_high,
        'Boot_Exp_CI_Lower': exp_low, 'Boot_Exp_CI_Upper': exp_high,
    })

df_ci = pd.DataFrame(results).set_index('Date')
print(df_ci)

## 2. Model-building approaches: EWMA and GARCH(1,1)

Two volatility estimators feeding the same normal-quantile VaR formula
(`VaR = -z * sigma`, z = 2.326 for 99%): EWMA with λ = 0.96, and a GARCH(1,1) fit by
maximum likelihood.

In [ ]:
lamda = 0.96

history = data.loc[:'2014-12-31']['Return']
new_return = new_data['Return']

last_var = np.var(history)
last_return = history.iloc[-1]

volatility = []
for i in new_return:
    current_var = lamda * last_var + (1 - lamda) * last_return**2
    volatility.append(np.sqrt(current_var))
    last_var, last_return = current_var, i

volatility = np.array(volatility)
VaR = -2.326 * volatility

ewma_results = pd.DataFrame({'EWMA VaR': VaR}, index=new_date)
print(ewma_results)
print(f'99% EWMA VaR: {np.percentile(VaR, 1)}')

In [ ]:
from arch import arch_model

returns = new_data['Return'] * 100

am = arch_model(returns, vol='Garch', p=1, q=1, mean='Zero', dist='Normal')
res = am.fit(disp='off')
daily_sigma = res.conditional_volatility / 100

z_score = 2.326
var = -z_score * daily_sigma

garch = pd.DataFrame({'Garch VaR': var})
print(garch)
print(f'99% GARCH VaR: {np.percentile(var, 1)}')

**Comparison.** Both approaches map volatility to VaR with the same normal quantile — the
difference is entirely in how volatility is estimated. GARCH VaR runs consistently more
conservative than EWMA and stays elevated longer at the end of the sample, consistent with
GARCH capturing volatility persistence (mean reversion in variance) that a pure exponential
decay does not. EWMA reacts faster to a single shock, but its decay can let VaR fall faster
than realized risk actually does — a full comparison needs backtesting, not just eyeballing
the series.

## 3. A volatility-normalized, EVT-based approach

Normalize daily gains by a rolling one-month volatility estimate — if the normalized series
looks closer to i.i.d., a VaR method can be built on the normalized residuals rather than
assuming a fixed unconditional distribution.

In [ ]:
normalize_return = []
for i in new_date:
    end_date = i - pd.Timedelta(days=1)
    start_date = i - pd.DateOffset(months=1)
    window_data = data[start_date:end_date]['Return']
    window_std = np.std(window_data)
    normalize_return.append(new_data.loc[i]['Return'] / window_std)

The normalized series has materially fatter tails than the raw returns visually, but a
comparable overall shape — consistent with volatility clustering being most of the story,
not a change in the underlying return distribution once volatility is accounted for.

**Filtered Historical Simulation via EVT.** Fit a Generalized Pareto distribution to the tail
of the *volatility-standardized* residuals (peaks-over-threshold, top 10%), then scale the
resulting quantile back up by the current conditional volatility.

In [ ]:
from scipy.stats import genpareto

data_aligned = pd.DataFrame({'Return': new_data['Return'], 'Sigma': volatility})
data_aligned.dropna(inplace=True)

returns_clean = data_aligned['Return']
sigma_clean = data_aligned['Sigma']
std_resid = returns_clean / sigma_clean

start_idx = 100 if len(returns_clean) > 150 else 20
evt_var_series = []

def calculate_evt_var_quantile(residuals, alpha=0.01, threshold_pct=0.10):
    losses = -residuals
    if len(losses) < 5:
        return np.nan
    u = np.quantile(losses, 1 - threshold_pct)
    excesses = losses[losses > u] - u
    if len(excesses) < 5:
        return -np.quantile(residuals, alpha)
    try:
        c, loc, scale = genpareto.fit(excesses, loc=0)
        N, Nu = len(losses), len(excesses)
        z_var = u + (scale / c) * (((N / Nu) * alpha) ** (-c) - 1)
        return -z_var
    except Exception:
        return -np.quantile(residuals, alpha)

for t in range(len(returns_clean)):
    if t < start_idx:
        evt_var_series.append(np.nan)
        continue
    history_z = std_resid.iloc[:t]
    z_quantile = calculate_evt_var_quantile(history_z.values, alpha=0.01)
    current_sigma = sigma_clean.iloc[t]
    evt_var_series.append(current_sigma * z_quantile)

data_aligned['EVT_VaR'] = evt_var_series
df_final = data_aligned.dropna()

comparison = pd.DataFrame(index=ewma_results.index)
comparison['Return'] = df_final['Return']
comparison['EVT_VaR'] = df_final['EVT_VaR']
common_index = comparison.index.intersection(ewma_results.index)
comparison = comparison.loc[common_index]
comparison['EWMA VaR'] = ewma_results.loc[common_index, 'EWMA VaR']

evt_exceptions = comparison[comparison['Return'] < comparison['EVT_VaR']]
ewma_exceptions = comparison[comparison['Return'] < comparison['EWMA VaR']]

total_days = len(comparison)
print(f"1. EWMA (Normal) Exceptions: {len(ewma_exceptions)} "
      f"({len(ewma_exceptions)/total_days:.2%})")
print(f"2. EVT (FHS)     Exceptions: {len(evt_exceptions)} "
      f"({len(evt_exceptions)/total_days:.2%})")

**Result:** the normal-tail EWMA approach produced 15 exceptions (2.01% — double the 1%
target). The EVT-based filtered historical simulation produced 5 exceptions (0.67%) — much
closer to, if anything slightly conservative relative to, the nominal coverage. Assuming
normal tails understates the true frequency of large losses; modeling the tail explicitly
with a generalized Pareto distribution corrects most of that gap.

**Recommendation to the head of trading:** use a real-time VaR framework that combines (1)
volatility scaling via EWMA or GARCH to adapt to changing market conditions, (2) explicit
tail modeling via extreme value theory rather than a normal-tail assumption, and (3) ongoing
backtesting with exception-rate escalation rules that trigger a methodology review if the
realized exception rate drifts too far from target.

## 4. Applying it live: 2-day 99% VaR ahead of an FOMC announcement

The same EWMA machinery, applied to live S&P 500 data, to size a 2-day 99% VaR for a $1M
position the day before a scheduled FOMC meeting. This part of the notebook runs standalone
against public data.

In [ ]:
import yfinance as yf
import numpy as np

sp500 = yf.download("^GSPC", start="2025-01-01", end="2026-02-01")['Close']
returns = sp500.pct_change().dropna()

lambda_param = 0.94
volatility_ewma = returns.ewm(alpha=1 - lambda_param, adjust=False).std()
current_daily_vol = float(volatility_ewma.iloc[-1])

capital = 10**6
z_score = 2.33
time_horizon = np.sqrt(2)  # 2-day horizon, square-root-of-time scaling

ewma_var_value = z_score * current_daily_vol * time_horizon * capital

print(f"Current Daily Volatility (EWMA): {current_daily_vol:.4%}")
print(f"Two-day 99% VaR: ${ewma_var_value:,.2f}")

**Result (as submitted):** current EWMA daily volatility ≈ 0.66%, giving a two-day 99%
VaR of **≈ $21,712** on a $1M position.

**On methodology going into the announcement:** a standard EWMA/normal-tail VaR assumes
market returns are approximately Gaussian, which the backtests above show understates true
tail risk — over the sample, the normal-tail approach exceeded its target exception rate by
roughly 2x. Ahead of a scheduled macro catalyst like an FOMC decision, that gap matters more,
not less: realized volatility tends to jump discontinuously around the announcement rather
than drift the way EWMA assumes. The EVT-based approach from Part 3 — scaling a fitted
generalized Pareto tail by current conditional volatility — is the more defensible choice for
sizing risk into a known event, since it doesn't rely on the normal-tail assumption that
back-testing shows is the weak point.